### Remove Images and Labels from the YOLO Dataset

This code removes selected images and their corresponding annotation files from the YOLO dataset. It processes the `train`, `val`, and `test` splits by reading the filenames listed in separate text files (`removed_files_train.txt`, `removed_files_val.txt`, and `removed_files_test.txt`).

For each filename, the script checks whether the corresponding `.jpg` image and `.txt` label exist. If they do, both files are deleted to keep the dataset images and annotations synchronized.

In [ ]:
from pathlib import Path

DATASET = Path("weapon-dataset-yolo/weapon-dataset-4f00d766")

List_txt = ["train", "val", "test"]

for split in List_txt:

    Image_dataset = DATASET / "images" / split
    Label_dataset = DATASET / "labels" / split

    list_of_images = Path(f"removed_files_{split}.txt")

    with open(list_of_images, "r") as f:
        list_of_files = f.readlines()

    for file_name in list_of_files:

        file_name = file_name.strip()

        if not file_name:
            continue

        Image_path = Image_dataset / f"{file_name}.jpg"
        Label_path = Label_dataset / f"{file_name}.txt"

        if Image_path.exists():
            Image_path.unlink()
            print(f"Deleted image: {Image_path}")

        if Label_path.exists():
            Label_path.unlink()
            print(f"Deleted label: {Label_path}")

### Filter Non-Real Images Using EfficientNet-B0

This code uses a pretrained EfficientNet-B0 image classification model to identify and filter out non-real images from the YOLO dataset. The model classifies each image into one of three categories: `anime`, `real`, or `rendered`.

The script processes the `train`, `val`, and `test` splits, resizing and normalizing each image before classification. Images predicted as `anime` or `rendered` are marked for removal only when the model's confidence is equal to or higher than the defined threshold (85%).

When an image is selected for removal, the script also deletes its corresponding YOLO annotation file to keep the dataset synchronized. The `DRY_RUN` variable is included as a configuration option for reviewing results before making changes.

In [ ]:
from PIL import Image
from torchvision import transforms
from pathlib import Path
import shutil
import torch
import timm
import torch

from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

DATASET = Path("weapon-dataset-yolo/weapon-dataset-4f00d766")

# Hugging Face repository
REPO_ID = "Mitchins/image-medium-classifier-efficientnet-b0-v1"

# Download model weights
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="model.safetensors"
)

# Create EfficientNet-B0 architecture
model = timm.create_model(
    "efficientnet_b0",
    num_classes=3,
    pretrained=False
)

# Load pretrained weights
state_dict = load_file(model_path)

model.load_state_dict(state_dict)

# Evaluation mode
model.eval()

print("Model loaded successfully!")

labels = ["anime", "real", "rendered"]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# Configuration
CONFIDENCE_THRESHOLD = 0.85
DRY_RUN = True  # Change to False after reviewing results

extensions = [".jpg", ".jpeg", ".png", ".webp"]

def classify_image(image_path):

    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0)

    with torch.inference_mode():
        output = model(tensor)
        probabilities = torch.softmax(output, dim=1)

    prediction = probabilities.argmax(dim=1).item()
    confidence = probabilities[0, prediction].item()

    return labels[prediction], confidence


# Process dataset
for split in ["train", "val", "test"]:

    images_dir = DATASET / "images" / split
    labels_dir = DATASET / "labels" / split

    for image_path in images_dir.iterdir():

        if image_path.suffix.lower() not in extensions:
            continue

        try:
            prediction, confidence = classify_image(image_path)

            print(
                f"{image_path.name}: "
                f"{prediction} ({confidence:.2%})"
            )

            # Only discard confident non-real predictions
            if prediction in ["rendered", "anime"] and confidence >= CONFIDENCE_THRESHOLD:

                print(f"TO DISCARD: {image_path.name}")

                # Delete image
                image_path.unlink(missing_ok=True)

                # Delete corresponding YOLO annotation
                label_path = labels_dir / f"{image_path.stem}.txt"

                label_path.unlink(missing_ok=True)

                print(f"Deleted image and label: {image_path.name}")

        except Exception as error:
            print(f"Error processing {image_path.name}: {error}")

print("Dataset filtering completed!")

## SigLIP2 – Zero-Shot Image Classification

This cell uses **SigLIP2** to identify potentially synthetic images in a weapon dataset. By comparing images against text descriptions, the model can distinguish between real photographs, video game screenshots, 3D renders, and digital illustrations without requiring additional training.

The goal is to identify suspicious images that may need to be removed from the dataset.

In [1]:

from pathlib import Path
from PIL import Image
from transformers import pipeline

# Dataset
DATASET = Path("weapon-dataset-yolo/weapon-dataset-4f00d766")

# Model
MODEL_NAME = "google/siglip2-base-patch16-224"

print("Loading SigLIP2...")

classifier = pipeline(
    task="zero-shot-image-classification",
    model=MODEL_NAME,
    device=-1  # CPU
)

print("Model loaded!")

# Descriptions
CANDIDATE_LABELS = [
    "a real photograph of a firearm in the real world",
    "a screenshot from a realistic video game showing a firearm",
    "a 3D rendered model of a firearm",
    "a digital illustration or drawing of a firearm"
]

# Configuration
THRESHOLD = 0.45

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def classify_image(image_path):

    with Image.open(image_path) as image:
        image = image.convert("RGB")

        results = classifier(
            image,
            candidate_labels=CANDIDATE_LABELS
        )

    return results


# Process dataset
for split in ["train", "val", "test"]:

    images_dir = DATASET / "images" / split

    if not images_dir.exists():
        continue

    for image_path in list(images_dir.iterdir()):

        if image_path.suffix.lower() not in EXTENSIONS:
            continue

        try:

            results = classify_image(image_path)

            best = results[0]

            print("\n", image_path.name)
            print("Prediction:", best["label"])
            print("Score:", round(best["score"], 4))

            # Detect suspicious images
            if (
                best["label"] != CANDIDATE_LABELS[0]
                and best["score"] >= THRESHOLD
            ):
                print("Suspicious image detected! Removing...")
                image_path.unlink()

        except Exception as error:
            print("Error:", image_path.name, error)

print("\nDataset analysis completed!")


Loading SigLIP2...


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
c:\Users\Tusab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tusab\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode 

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\Tusab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tusab\.cache\huggingface\hub\models--google--siglip2-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

Model loaded!

 0001d500af5f8519e3f92170d47e0331_2578.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.0104

 0003db56eac6fde6b0959df7f1cc14e1_2579.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.0

 000750efce272d1a6c78a4be51675e4d_2580.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.0

 000ec5ab9bfe5c5b4821621cca494150_2583.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.0191

 0015e9d4a3ddfb2f302609c3c70d849f_2584.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.1178

 0017fb67e20ce4a45eb072badbf44696_2585.jpg
Prediction: a 3D rendered model of a firearm
Score: 0.0001

 00180587b5b10680c9ae31781a3ae3ea_2586.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.0

 001b5fbdc1068b222a18d6689c13a213_2588.jpg
Prediction: a real photograph of a firearm in the real world
Score: 0.0007

 001be8f019c841978ae2127a8cf95b5a_2589.jpg
Prediction: a 3